# Лабораторная работа № 3.5 — Интегрирование и метод Рунге–Ромберга

Вариант 6. 

Считаем интеграл тремя квадратурными формулами на шагах 0.5 и 0.25, затем оцениваем и уточняем результат по разнице сеток.

## Исходные данные

Числа ниже соответствуют файлу input.txt этой работы.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from typing import List

rows = [[-1.0, 1.0], [0.5, 0.25]]
(left, right), (h1, h2) = rows
print('Отрезок:', (left, right), 'шаги:', (h1, h2))

Отрезок: (-1.0, 1.0) шаги: (0.5, 0.25)


## Три квадратурные формулы

Прямоугольники используют середины, трапеции — концы промежутков, Симпсон — чередующиеся веса 4 и 2.

In [2]:
def f(x):
    return x / ((2 * x + 7) * (3 * x + 4))


def _interval_count(l, r, h):
    if not all(math.isfinite(value) for value in (l, r, h)) or h <= 0 or r <= l:
        raise ValueError("Require finite l < r and positive h")
    count = round((r - l) / h)
    if count < 1 or not math.isclose(count * h, r - l, rel_tol=1e-12, abs_tol=1e-12):
        raise ValueError("Step must divide the integration interval")
    return count


def integrate_rectangle_method(f, l, r, h):
    count = _interval_count(l, r, h)
    return h * sum(f(l + (i + 0.5) * h) for i in range(count))


def integrate_trapeze_method(f, l, r, h):
    count = _interval_count(l, r, h)
    return h * (0.5 * (f(l) + f(r)) + sum(f(l + i * h) for i in range(1, count)))


def integrate_simpson_method(f, l, r, h):
    count = _interval_count(l, r, h)
    if count % 2:
        raise ValueError("Simpson's rule requires an even number of intervals")
    weighted_sum = sum((4 if i % 2 else 2) * f(l + i * h) for i in range(1, count))
    return h / 3 * (f(l) + weighted_sum + f(r))

In [3]:
methods = [('Прямоугольники', integrate_rectangle_method, 2), ('Трапеции', integrate_trapeze_method, 2), ('Симпсон', integrate_simpson_method, 4)]
results = []
for name, method, order in methods:
    coarse = method(f, left, right, h1)
    fine = method(f, left, right, h2)
    results.append((name, coarse, fine, order))
    print(name, 'h1:', coarse, 'h2:', fine)

Прямоугольники h1: -0.034310603265116243 h2: -0.039242405208091885
Трапеции h1: -0.05701659451659452 h2: -0.04566359889085538
Симпсон h1: -0.04533429533429534 h2: -0.04187926701560901


## Уточнение

Поправка к значению на мелкой сетке равна (I_h2−I_h1)/(2^p−1). Для прямоугольников и трапеций p=2, для Симпсона p=4.

In [4]:
def runge_rombert_method(h1, h2, integral1, integral2, p):
    """Поправка к интегралу на мелкой сетке и уточнённое значение."""
    if h1 > h2:
        coarse_h, fine_h, coarse, fine = h1, h2, integral1, integral2
    else:
        coarse_h, fine_h, coarse, fine = h2, h1, integral2, integral1
    correction = (fine - coarse) / ((coarse_h / fine_h)**p - 1)
    return correction, fine + correction

In [5]:
refined = {}
for name, coarse, fine, order in results:
    correction, estimate = runge_rombert_method(h1, h2, coarse, fine, order)
    refined[name] = estimate
    print(name, 'поправка:', correction, 'уточнённый интеграл:', estimate)

Прямоугольники поправка: -0.0016439339809918806 уточнённый интеграл: -0.04088633918908376
Трапеции поправка: 0.0037843318752463794 уточнённый интеграл: -0.041879267015609005
Симпсон поправка: 0.00023033522124575508 уточнённый интеграл: -0.041648931794363256


## Самопроверка

Сравни уточнённые значения с точным интегралом, вычисленным по первообразной.

In [6]:
F = lambda x: 7/26*math.log(abs(2*x+7))-4/39*math.log(abs(3*x+4))
exact = F(right)-F(left)
errors = {name: abs(value-exact) for name, value in refined.items()}
assert min(errors, key=errors.get) == 'Симпсон'
print('Точное значение:', exact)
print('Ошибки:', errors)

Точное значение: -0.0413302721730513
Ошибки: {'Прямоугольники': 0.0004439329839675335, 'Трапеции': 0.0005489948425577082, 'Симпсон': 0.00031865962131195913}
